In [1]:
def call_cost(I, O, C=0, p_i=0.30, p_o=1.20, p_cache=0.006,
              batch=False, offpeak=False, offpeak_ratio=0.5):
    """C: 命中缓存的输入 token 数；I 需 >= C"""
    assert I >= C, "缓存命中的 token 数不能超过总输入"
    if offpeak:                      # 低谷价按厂商定义为标准价的一半
        p_i, p_o, p_cache = p_i * offpeak_ratio, p_o * offpeak_ratio, p_cache * offpeak_ratio
    cost = (C * p_cache + (I - C) * p_i + O * p_o) / 1e6
    if batch:                        # Batch API 输入输出均五折
        cost *= 0.5
    return cost

# 例：批量抽取任务，固定系统提示 8K token 全部命中缓存，单条正文 2K 输入，输出 300 token
per_call = call_cost(I=2000, O=300, C=0, p_i=0.30, p_o=1.20, batch=True)
print(f"单条批处理成本: ${per_call:.6f}")          # 约 3.3e-4 美元
print(f"十万条总成本:   ${per_call * 1e5:.2f}")

# 对比：同一任务走标准同步接口、且输出膨胀到 1500 token（例如要求给出推理过程）
per_call_sync = call_cost(I=2000, O=1500, C=0, p_i=0.30, p_o=1.20)
print(f"单条同步+长输出: ${per_call_sync:.6f}")    # 输出侧主导成本
print(f"放大倍数: {per_call_sync / per_call:.1f}x")

单条批处理成本: $0.000480
十万条总成本:   $48.00
单条同步+长输出: $0.002400
放大倍数: 5.0x


In [2]:

def breakeven_hours(P, r, e=0.36, T=3, salvage_rate=0.0):
    """P: 采购价；r: 云租每小时价；e: 自购每小时运行成本；T: 持有年限
       salvage_rate: 期末残值占采购价的比例"""
    net_cost = P * (1 - salvage_rate)          # 扣除残值后的净投入
    return net_cost / ((r - e) * T)

P, e, T = 31500, 0.36, 3
for r in (3, 4, 5, 5.2, 2.8):
    h = breakeven_hours(P, r, e, T)
    print(f"云租 ￥{r:>4}/时 -> 平衡年机时 {h:7.0f} 小时，折合每日 {h/365:5.1f} 小时")

# 敏感性：残值率每提高 10 个百分点，平衡机时下降同等比例
for sr in (0.0, 0.2, 0.4):
    h = breakeven_hours(P, 4.0, e, T, salvage_rate=sr)
    print(f"残值率 {sr:.0%} -> 平衡年机时 {h:.0f} 小时（每日 {h/365:.1f} 小时）")

云租 ￥   3/时 -> 平衡年机时    3977 小时，折合每日  10.9 小时
云租 ￥   4/时 -> 平衡年机时    2885 小时，折合每日   7.9 小时
云租 ￥   5/时 -> 平衡年机时    2263 小时，折合每日   6.2 小时
云租 ￥ 5.2/时 -> 平衡年机时    2169 小时，折合每日   5.9 小时
云租 ￥ 2.8/时 -> 平衡年机时    4303 小时，折合每日  11.8 小时
残值率 0% -> 平衡年机时 2885 小时（每日 7.9 小时）
残值率 20% -> 平衡年机时 2308 小时（每日 6.3 小时）
残值率 40% -> 平衡年机时 1731 小时（每日 4.7 小时）


In [3]:
import numpy as np

P, e, T = 31_500.0, 0.36, 3.0
h = np.linspace(0, 4500, 4501)          # 年机时网格（上限须覆盖最大交点）

for r in (3.0, 4.0, 5.0):
    own = P + e * h * T
    rent = r * h * T
    cross = h[np.argmin(np.abs(own - rent))]
    print(f"r=￥{r:.0f}/时: 交点年机时 {cross:6.0f} 小时（每日 {cross/365:4.1f} 小时），"
          f"交点处两条曲线成本 {P + e*cross*T:8.0f} 元")
    # 数值解与解析解 h* = P/((r-e)T) 对照
    analytic = P / ((r - e) * T)
    print(f"            解析解 {analytic:6.0f} 小时，网格解与解析解之差 {abs(cross-analytic):.2f} 小时")

# 敏感性：r=4 时云租价、采购价、残值率各变动 ±30%，交点如何移动
r = 4.0
def crossing(P_, r_, e_, salvage=0.0):
    return P_ * (1 - salvage) / ((r_ - e_) * T)

base = crossing(P, r, e)
print("\n敏感性（r=4，基准平衡机时 %.0f 小时/年）:" % base)
for name, lo, hi in [("云租价 r ±30%", crossing(P, r*0.7, e), crossing(P, r*1.3, e)),
                     ("采购价 P ±30%",  crossing(P*0.7, r, e), crossing(P*1.3, r, e)),
                     ("残值率 0/40%",   crossing(P, r, e, 0.0), crossing(P, r, e, 0.4))]:
    print(f"  {name}: 平衡机时 [{lo:6.0f}, {hi:6.0f}] 小时/年")

r=￥3/时: 交点年机时   3977 小时（每日 10.9 小时），交点处两条曲线成本    35795 元
            解析解   3977 小时，网格解与解析解之差 0.27 小时
r=￥4/时: 交点年机时   2885 小时（每日  7.9 小时），交点处两条曲线成本    34616 元
            解析解   2885 小时，网格解与解析解之差 0.38 小时
r=￥5/时: 交点年机时   2263 小时（每日  6.2 小时），交点处两条曲线成本    33944 元
            解析解   2263 小时，网格解与解析解之差 0.07 小时

敏感性（r=4，基准平衡机时 2885 小时/年）:
  云租价 r ±30%: 平衡机时 [  4303,   2169] 小时/年
  采购价 P ±30%: 平衡机时 [  2019,   3750] 小时/年
  残值率 0/40%: 平衡机时 [  2885,   1731] 小时/年


In [4]:
GB = 1024 ** 3

def infer_vram(params_b, bytes_per_param=2.0, seq_len=8192, batch=1,
               kv_bytes_per_token=131072, overhead=1.2):
    """params_b: 参数量（十亿）；kv_bytes_per_token: 每 token 的 KV 字节数"""
    weights = params_b * 1e9 * bytes_per_param * overhead
    kv = kv_bytes_per_token * seq_len * batch
    return weights / GB, kv / GB, (weights + kv) / GB

for name, p, b, kvpt in [("7B BF16", 7, 2.0, 57344),
                         ("8B BF16", 8, 2.0, 131072),
                         ("14B BF16", 14, 2.0, 131072),
                         ("32B Q4_K_M", 32, 0.55, 131072),
                         ("70B Q4_K_M", 70, 0.55, 131072)]:
    w, k, t = infer_vram(p, b, kv_bytes_per_token=kvpt)
    print(f"{name:>12}: 权重 {w:6.2f} GB + KV {k:5.2f} GB = {t:6.2f} GB")

# 同一模型拉长上下文：KV 项线性增长，权重项不变
for s in (2048, 8192, 32768, 131072):
    _, k, t = infer_vram(8, 2.0, seq_len=s)
    print(f"8B @ seq={s:>6}: KV {k:5.2f} GB, 合计 {t:6.2f} GB")

     7B BF16: 权重  15.65 GB + KV  0.44 GB =  16.08 GB
     8B BF16: 权重  17.88 GB + KV  1.00 GB =  18.88 GB
    14B BF16: 权重  31.29 GB + KV  1.00 GB =  32.29 GB
  32B Q4_K_M: 权重  19.67 GB + KV  1.00 GB =  20.67 GB
  70B Q4_K_M: 权重  43.03 GB + KV  1.00 GB =  44.03 GB
8B @ seq=  2048: KV  0.25 GB, 合计  18.13 GB
8B @ seq=  8192: KV  1.00 GB, 合计  18.88 GB
8B @ seq= 32768: KV  4.00 GB, 合计  21.88 GB
8B @ seq=131072: KV 16.00 GB, 合计  33.88 GB


In [5]:
GB = 1024 ** 3

def full_ft_vram(params_b, opt_bits=32, master_fp32=True):
    """16 字节/参数 = bf16 权重 2 + bf16 梯度 2 + fp32 主权重 4 + m 4 + v 4
       opt_bits=8 时把两个动量项降为 8-bit"""
    per = 2 + 2 + (4 if master_fp32 else 0)
    per += (4 + 4) if opt_bits == 32 else (1 + 1)
    return params_b * 1e9 * per / GB

def lora_vram(params_b, base_bytes=2.0, adapter_ratio=0.01, activation_gb=4.0):
    base = params_b * 1e9 * base_bytes / GB
    # 适配器参数+其 AdamW 状态：可训练参数按 16 字节计，反向还需基座梯度（此处按冻结处理）
    adapter = params_b * 1e9 * adapter_ratio * 16 / GB
    return base + adapter + activation_gb

for p in (7, 13, 32, 70):
    print(f"{p:>3}B 全参(AdamW fp32 动量): {full_ft_vram(p):7.1f} GB | "
          f"8-bit 优化器: {full_ft_vram(p, opt_bits=8):7.1f} GB")

for p, b in ((7, 0.55), (7, 2.0), (13, 0.55), (32, 0.55), (70, 0.55)):
    print(f"{p:>3}B 基座 {b} 字节/参数 -> LoRA/QLoRA 估算 {lora_vram(p, b):6.1f} GB "
          f"（含 {4.0} GB 激活假设）")

  7B 全参(AdamW fp32 动量):   104.3 GB | 8-bit 优化器:    65.2 GB
 13B 全参(AdamW fp32 动量):   193.7 GB | 8-bit 优化器:   121.1 GB
 32B 全参(AdamW fp32 动量):   476.8 GB | 8-bit 优化器:   298.0 GB
 70B 全参(AdamW fp32 动量):  1043.1 GB | 8-bit 优化器:   651.9 GB
  7B 基座 0.55 字节/参数 -> LoRA/QLoRA 估算    8.6 GB （含 4.0 GB 激活假设）
  7B 基座 2.0 字节/参数 -> LoRA/QLoRA 估算   18.1 GB （含 4.0 GB 激活假设）
 13B 基座 0.55 字节/参数 -> LoRA/QLoRA 估算   12.6 GB （含 4.0 GB 激活假设）
 32B 基座 0.55 字节/参数 -> LoRA/QLoRA 估算   25.2 GB （含 4.0 GB 激活假设）
 70B 基座 0.55 字节/参数 -> LoRA/QLoRA 估算   50.3 GB （含 4.0 GB 激活假设）


In [6]:
GB = 1024 ** 3

def train_vram_breakdown(params_b, layers, hidden, heads,
                         seq_len=2048, batch=1,
                         grad_checkpointing=False):
    per_param = 2 + 2 + 4 + 4 + 4           # bf16 权重+梯度, fp32 主权重, m, v
    static = params_b * 1e9 * per_param / GB
    if grad_checkpointing:
        # 只保存各层分段边界输入：L * B * S * h * 2 字节
        act = layers * batch * seq_len * hidden * 2 / GB
        act_label = "仅层边界输入（检查点）"
    else:
        per_token = hidden * (34 + 5 * heads * seq_len / hidden)
        act = layers * batch * seq_len * per_token / GB
        act_label = "全量激活（无检查点）"
    return static, act, act_label

# 7B 级模型典型形状：L=32, h=4096, 注意力头 a=32
for ckpt in (False, True):
    static, act, label = train_vram_breakdown(7, 32, 4096, 32,
                                              seq_len=2048, batch=1,
                                              grad_checkpointing=ckpt)
    print(f"7B 全参微调 @S=2048,B=1 [{label}]: "
          f"权重+梯度+优化器 {static:6.1f} GiB + 激活 {act:5.1f} GiB = {static+act:6.1f} GiB")

# 与 19.10.1 的 16 字节/参数静态项对照：静态项不含激活
print(f"16 字节/参数静态项核对: 7e9*16/GB = {7e9*16/GB:.1f} GiB（约 112 GB 十进制）")

7B 全参微调 @S=2048,B=1 [全量激活（无检查点）]: 权重+梯度+优化器  104.3 GiB + 激活  28.5 GiB =  132.8 GiB
7B 全参微调 @S=2048,B=1 [仅层边界输入（检查点）]: 权重+梯度+优化器  104.3 GiB + 激活   0.5 GiB =  104.8 GiB
16 字节/参数静态项核对: 7e9*16/GB = 104.3 GiB（约 112 GB 十进制）


In [7]:
import numpy as np

def optimal_scale(c0, kappa, tau2, A):
    """返回最优单次规模 s*、对应重复次数与最小标准误系数"""
    s = np.sqrt(c0 * A / (kappa * tau2))
    return s

def se_of(s, B, c0, kappa, tau2, A):
    n = B / (c0 + kappa * s)
    return np.sqrt(tau2 / n + A / (n * s))

B, c0, kappa = 3000.0, 60.0, 0.35     # 预算 3000 元；固定开销 60 元；每单位规模 0.35 元
for tau2, A, label in [(0.04, 4.0, "训练稳定、评测噪声大"),
                       (0.25, 4.0, "训练不稳定、评测噪声大"),
                       (0.25, 0.5, "训练不稳定、评测噪声小")]:
    s_star = optimal_scale(c0, kappa, tau2, A)
    n_star = B / (c0 + kappa * s_star)
    print(f"{label}: s*={s_star:8.1f}, n*={n_star:5.1f}, "
          f"SE={se_of(s_star, B, c0, kappa, tau2, A):.4f}")

# 验证：在 s* 附近网格搜索，数值解应与解析解一致
tau2, A = 0.25, 4.0
grid = np.linspace(1, 400, 4000)
best = min(grid, key=lambda s: se_of(s, B, c0, kappa, tau2, A))
print(f"解析解 s*={optimal_scale(c0, kappa, tau2, A):.2f}，"
      f"网格搜索 s*={best:.2f}")

# 预算翻倍对标准误的影响
for Bv in (1500, 3000, 6000, 12000):
    s = optimal_scale(c0, kappa, tau2, A)
    print(f"预算 {Bv:>6}: SE={se_of(s, Bv, c0, kappa, tau2, A):.4f}"
          f"（相对 1500 时为 {se_of(s,Bv,c0,kappa,tau2,A)/se_of(s,1500,c0,kappa,tau2,A):.3f}）")

训练稳定、评测噪声大: s*=   130.9, n*= 28.3, SE=0.0499
训练不稳定、评测噪声大: s*=    52.4, n*= 38.3, SE=0.0923
训练不稳定、评测噪声小: s*=    18.5, n*= 45.1, SE=0.0783
解析解 s*=52.37，网格搜索 s*=52.38
预算   1500: SE=0.1306（相对 1500 时为 1.000）
预算   3000: SE=0.0923（相对 1500 时为 0.707）
预算   6000: SE=0.0653（相对 1500 时为 0.500）
预算  12000: SE=0.0462（相对 1500 时为 0.354）


In [8]:
import numpy as np

rng = np.random.default_rng(20260924)
# 示例：预实验记录的 25 次 7B QLoRA 微调壁钟时间（小时）
times = np.array([8.2, 7.6, 9.1, 8.8, 7.9, 10.3, 8.1, 8.5, 7.7, 9.4, 8.0, 8.9,
                  11.2, 7.8, 8.3, 9.0, 8.6, 7.5, 8.4, 12.1, 8.7, 9.2, 8.1, 7.9, 8.8])

mu, sd = times.mean(), times.std(ddof=1)
hourly, n = 2.4, 15                      # 云租单价（元/时）与计划运行次数
print(f"单次机时: 均值 {mu:.2f} h, 标准差 {sd:.2f} h, 变异系数 {sd/mu:.3f}")

mean_budget = n * mu * hourly
quant_budget = (n * mu + 1.645 * np.sqrt(n) * sd) * hourly
print(f"均值口径预算: {mean_budget:.0f} 元；95% 分位口径预算: {quant_budget:.0f} 元")

# bootstrap：对 n 次运行的总机时做重抽样，直接取 95% 分位数
boot = rng.choice(times, size=(20000, n), replace=True).sum(axis=1) * hourly
print(f"bootstrap 95% 分位预算: {np.quantile(boot, 0.95):.0f} 元；"
      f"中位数: {np.median(boot):.0f} 元")

单次机时: 均值 8.72 h, 标准差 1.10 h, 变异系数 0.126
均值口径预算: 314 元；95% 分位口径预算: 331 元
bootstrap 95% 分位预算: 332 元；中位数: 313 元


In [9]:
import numpy as np

rng = np.random.default_rng(19)
N = 200_000

# 参数的不确定性设定（均为示意，需用实测数据替换）
P      = rng.normal(31_500, 2_000, N)        # 采购价
r      = rng.uniform(3.0, 5.0, N)            # 云租小时价
e      = rng.normal(0.36, 0.05, N)           # 每小时电费与维护
T      = 3.0                                 # 持有年限
h_daily= rng.uniform(4, 10, N)               # 日均使用机时
h_year = h_daily * 365

own  = P + e * h_year * T                    # 自购总成本
rent = r * h_year * T                        # 云租总成本
p_own_cheaper = np.mean(own < rent)
print(f"自购更省的概率: {p_own_cheaper:.1%}")

# 决策翻转的阈值：找出使 own == rent 的云租价
r_star = P / (h_year * T) + e
print(f"盈亏平衡云租价: 中位数 {np.median(r_star):.2f} 元/时，"
      f"5% 分位 {np.quantile(r_star, 0.05):.2f}，95% 分位 {np.quantile(r_star, 0.95):.2f}")

# API 单价 ±30% 对盈亏平衡调用量的影响（单位统一为元与百万 token）
F, p0, v = 20_000, 2.10, 0.15      # 年固定成本、API 单价、自建单位边际成本
base = F / (p0 - v)
for delta in (-0.3, 0.0, 0.3):
    p = p0 * (1 + delta)
    q = F / (p - v)
    print(f"API 单价 ￥{p:.2f}/百万 token（{delta:+.0%}）-> 平衡调用量 {q:,.0f} 百万 token"
          f"（相对基准 {q/base:.1%}），弹性 {-p/(p-v):.2f}")

自购更省的概率: 32.6%
盈亏平衡云租价: 中位数 4.47 元/时，5% 分位 3.26，95% 分位 7.09
API 单价 ￥1.47/百万 token（-30%）-> 平衡调用量 15,152 百万 token（相对基准 147.7%），弹性 -1.11
API 单价 ￥2.10/百万 token（+0%）-> 平衡调用量 10,256 百万 token（相对基准 100.0%），弹性 -1.08
API 单价 ￥2.73/百万 token（+30%）-> 平衡调用量 7,752 百万 token（相对基准 75.6%），弹性 -1.06


In [10]:
import numpy as np

rng = np.random.default_rng(2026)
theta, tau = 0.72, 0.03          # 真实指标与运行间标准差（示意）

def simulate(n, reps=20000):
    Y = rng.normal(theta, tau, size=(reps, n))
    mean = Y.mean(axis=1)
    se   = Y.std(axis=1, ddof=1) / np.sqrt(n)
    ci_half = 4.303 * se if n == 3 else 1.96 * se
    return mean.mean(), (mean - ci_half < theta).mean(), Y.max(axis=1).mean()

for n in (3, 5, 10, 20):
    m, cov, mx = simulate(n)
    print(f"n={n:>2}: 均值 {m:.4f}（真值 {theta}），区间覆盖率约 {cov:.2f}，"
          f"取最大值的平均报告值 {mx:.4f}（偏差 +{mx-theta:.4f}）")

n= 3: 均值 0.7200（真值 0.72），区间覆盖率约 0.97，取最大值的平均报告值 0.7454（偏差 +0.0254）
n= 5: 均值 0.7201（真值 0.72），区间覆盖率约 0.94，取最大值的平均报告值 0.7550（偏差 +0.0350）
n=10: 均值 0.7201（真值 0.72），区间覆盖率约 0.96，取最大值的平均报告值 0.7662（偏差 +0.0462）
n=20: 均值 0.7200（真值 0.72），区间覆盖率约 0.97，取最大值的平均报告值 0.7760（偏差 +0.0560）


In [11]:
import numpy as np

rng = np.random.default_rng(7)
N = 5000
t = rng.uniform(1, 12, N)                     # 运行壁钟时间（小时）
Y = 0.6 + 0.02 * t + rng.normal(0, 0.05, N)   # 结果与时长的真实关系：跑得久略好
pi = np.clip(1.05 - 0.05 * t, 0.15, 1.0)      # 完成概率随时长下降
R = (rng.random(N) < pi).astype(int)

naive = Y[R == 1].mean()                      # 只分析完成的运行
ipw_num = (R * Y / pi).sum(); ipw_den = (R / pi).sum()
ipw = ipw_num / ipw_den
truth = Y.mean()

print(f"全体真值 {truth:.4f} | 仅完成样本 {naive:.4f} | IPW 校正 {ipw:.4f}")
print(f"朴素估计偏倚 {naive - truth:+.4f}；IPW 估计偏倚 {ipw - truth:+.4f}")
print(f"建议报告的敏感性区间：[{min(naive, ipw):.4f}, {max(naive, ipw):.4f}]")

全体真值 0.7292 | 仅完成样本 0.7147 | IPW 校正 0.7288
朴素估计偏倚 -0.0144；IPW 估计偏倚 -0.0004
建议报告的敏感性区间：[0.7147, 0.7288]


In [12]:
import numpy as np

rng = np.random.default_rng(19)
N = 200_000
t = rng.uniform(1, 12, N)                    # 计划任务时长（小时）
pi = np.clip(1.05 - 0.05 * t, 0.15, 1.0)     # 单次尝试的完成概率，随时长下降
R = (rng.random(N) < pi)                     # 单次尝试是否完成

# (1) 选择偏倚：完成样本的时长分布系统性偏短
print(f"全体任务平均时长 {t.mean():.2f} h | 完成任务平均时长 {t[R].mean():.2f} h | "
      f"完成率 {R.mean():.1%}")

# (2) 成本折算：无检查点时，时长为 t 的任务期望耗机 t/pi(t)（几何重试）
expected_machine = (t / pi).mean()           # 每完成一个任务的期望机时
naive = t[R].mean()                          # 朴素估计：只看完成样本、忽略重试
hourly = 2.4
print(f"朴素成本估计（完成样本均值×单价）: {naive*hourly:5.2f} 元/任务")
print(f"按重试折算的期望成本（E[t/pi]×单价）: {expected_machine*hourly:5.2f} 元/任务")
print(f"低估幅度: {(1 - naive/expected_machine):.1%}")

# (3) 高频检查点下：失败仅损失自上次检查点以来的进度（近似为检查点间隔 delta 的一半），
#     期望耗机 ≈ t + (1/pi - 1) * delta/2，重试折算的增量大幅缩小
delta = 0.5
expected_ckpt = (t + (1 / pi - 1) * delta / 2).mean()
print(f"检查点（间隔 {delta} h）折算后的期望成本: {expected_ckpt*hourly:5.2f} 元/任务")

全体任务平均时长 6.51 h | 完成任务平均时长 5.81 h | 完成率 72.5%
朴素成本估计（完成样本均值×单价）: 13.94 元/任务
按重试折算的期望成本（E[t/pi]×单价）: 25.20 元/任务
低估幅度: 44.7%
检查点（间隔 0.5 h）折算后的期望成本: 15.89 元/任务


In [13]:
q = 0.15                    # 中断重跑率
gpu_hours = 9.0             # 单次机时（小时）
price = 2.4                 # 云租单价（元/时）
storage, transfer = 0.5, 1.6

gpu = gpu_hours * price
retry = gpu * q / (1 - q)   # 无检查点假设下的重试折算
buffer = 0.2 * (gpu + storage + transfer + retry)
items = [("GPU 机时", gpu), ("存储", storage), ("传输", transfer),
         ("重试折算", retry), ("缓冲", buffer)]
for name, v in items:
    print(f"{name}: {v:5.2f} 元")
print(f"期望总成本: {sum(v for _, v in items):5.2f} 元")

GPU 机时: 21.60 元
存储:  0.50 元
传输:  1.60 元
重试折算:  3.81 元
缓冲:  5.50 元
期望总成本: 33.01 元
